# Accessing Instantaneous Positions

**Version:** 1.0 | **Last updated:** 2026-07-14 

**Author:** Eshanta Mishra | **Author institution:** EarthScope Consortium

**Maintainer:** EarthScope OnRamp Team | **Maintainer's contact :** help@earthscope.org

**Estimated Time:**  ~ 45 minutes | **Pathway:** MVP1

**License:** CC-BY-4.0

## Introduction

**What this notebook does:** The notebook retrieves Instantaneous (high-rate PPP stream) GNSS positions as dataframes for selected stations and time ranges using the EarthScope SDK, and produces visualizations of the position streams and the displacement derived from them.

**Why it is useful:** Instantaneous positions tell us where a station is right now, epoch by epoch. While daily solutions are useful for describing slow tectonic motion, this high-rate stream can capture ground movement that happens in seconds and provides a foundation for studying transient events like earthquakes.

**What you will accomplish:** By the end, you will have retrieved instantaneous positions for one or more stations, inspected and understood data fields that come with the instantaneous positions, derived 2D and 3D displacement magnitudes, and visualized both the position streams and displacement over time.

---

### Prerequisites

Before starting this notebook, you should:

* [ ] Have completed: [Notebook 1 - Accessing GNSS Observations with the EarthScope SDK](NB1-access-gnss-via-SDK.ipynb).
* [ ] Be familiar with basic Python.

---

### GeoLab Compute Resources

| Setting | Recommended |
|---|---|
| **Image** | GeoLab (default image) |
| **Server size** | 4 GB RAM, ~0.5 CPUs (default server) |

## Learning Objectives

By the end of this notebook, you will be able to:

1. Retrieve instantaneous (PPP) positions for one or more stations by station name, time range, facility, and software.
2. Load the result into a dataframe and interpret every returned field, including position components and their uncertainties.
3. Derive 2D and 3D displacement magnitudes from the east/north/up components.
4. Visualize position streams and displacement over time.

## Relevant Documentation & Resources

* [EarthScope SDK documentation](https://docs.earthscope.org/sdk)
* [SDK GNSS Observations tutorial](https://docs.earthscope.org/sdk/gnss-obs-tutorial)
* [Polars User Guide](https://docs.pola.rs)
* [Altair (plotting)](https://altair-viz.github.io)

## Contents

1. [What are Instantaneous Positions?](#id-1-what-are-instantaneous-positions)
2. [Setup & Imports](#id-2-setup-imports)
3. [Retrieve Instantaneous Positions](#id-3-retrieve-instantaneous-positions)
4. [Inspect the Returned Fields](#id-4-inspect-the-returned-fields)
5. [Derive Displacement](#id-5-derive-displacement)
6. [Visualize Position Streams & Displacement](#id-6-visualize-position-streams-displacement)
7. [Exploration Exercises](#id-7-exploration-exercises)
8. [Troubleshooting & Support](#id-8-troubleshooting-support)

## 1. What are Instantaneous Positions?

A GNSS station's position can be estimated in different ways depending on how much data goes into each estimate.

A very common geodetic product is a **daily position time series** which provides one position per station per day, formed by combining a full 24 hours of observations. Averaging over a day suppresses the noise down to millimeter precision. Stacking those daily positions over years yields the velocity fields that reveal slow tectonic motion such as plate drift and interseismic strain at the level of a few millimeters per year. The tradeoff of this precision is the time resolution because one point per day cannot be used to show anything that happens in seconds.

**Instantaneous positions** make the opposite tradeoff. Using Precise Point Positioning (PPP), the station's position is estimated epoch by epoch (once per second in this case). So you can see the current position of the stations continously. Individual estimates are far noisier than daily solutions. However, the high sampling rate makes it possible to detect rapid, transient motion in real time, such as an earthquake displacing a station within seconds. A daily solution would instead combine that motion into a single averaged position.

> **Note:** Derived daily position time series are not yet available through the SDK; this notebook focuses on the instantaneous (high-rate PPP) stream, which is already available.

## 2. Setup & Imports

In [ ]:
# Standard library imports
import datetime as dt

# Third-party imports
import altair as alt
import polars as pl
from earthscope_sdk import AsyncEarthScopeClient

# Enable the Rust (vegafusion) backend so Altair can handle larger datasets efficiently
alt.data_transformers.enable("vegafusion")

es = AsyncEarthScopeClient()

Polars' `.plot` uses Altair under the hood, with vegafusion as its fast backend.

### Configuration

Set your parameters here before running the rest of the notebook.

In [ ]:
# Modify these values before running the notebook.

STATIONS = ["P146", "P147", "P148"]      # one or more station IDs
FACILITY = "cwu"                         # analysis center producing the stream
SOFTWARE = "fastlane"                    # PPP software producing the stream
META_FIELDS = ["geosncl"]                # extra stream-metadata columns to attach
START = dt.datetime(2026, 6, 4, 10)      # query start (UTC)
END = dt.datetime(2026, 6, 4, 12)        # query end (UTC)

## 3. Retrieve Instantaneous Positions

In this step, we will retrieve high-rate PPP position stream for one or more stations, returned as an Apache Arrow table.

As with observations on [Notebook 1](NB1-access-gnss-via-SDK.ipynb), the SDK returns Arrow, which converts into a dataframe with little or no copying. The stream is selected not just by station and time, but also by the processing pipeline that produced it such as the analysis `facility` and the `software`.

Each argument below narrows what you get:

* `station_name`: one station ID or a list
* `facility`: the analysis center producing the stream (here `"cwu"`)
* `software`: the PPP engine (here `"fastlane"`)
* `meta_fields`: extra stream-metadata columns to attach (here `"geosncl"`, the stream identifier)

The expected result is roughly 21,600 rows, i.e. 3 stations x 2 hours x 3600 seconds, since the stream is 1 Hz.

In [ ]:
# Fetch a 2-hour window of 1 Hz instantaneous positions for the selected stations.
table = await es.data.gnss_instantaneous_positions(
    start_datetime=START,
    end_datetime=END,
    station_name=STATIONS,
    facility=FACILITY,
    software=SOFTWARE,
    meta_fields=META_FIELDS,
).fetch()

df = pl.from_arrow(table).sort("timestamp")
print(f"{len(df):,} rows")
df.head()

## 4. Inspect the Returned Fields

Get to know the data before analyzing it: which columns came back, how often the stream samples, which stations are present, and where values are missing.

In [ ]:
print("Columns:", df.columns)
print("Streams:", df["geosncl"].unique().sort().to_list())

### Sampling Interval

Look at the spacing between consecutive epochs for a single stream. It should be 1 seconds.

In [ ]:
one = df.filter(pl.col("geosncl").str.starts_with(STATIONS[0])).sort("timestamp")
one.select(pl.col("timestamp").diff().alias("dt"))["dt"].drop_nulls().value_counts(sort=True).head()

> Note: For a 1 Hz stream, any gap larger than 1s means epochs are missing from the table. For example, 2s is one dropped epoch, 3s is two, and so on. A few of these is normal for a real-time stream.

### Missing values

`east` and `north` are occasionally null while `up` is populated. Count nulls per column before deriving anything below. The square root propagates nulls, so those epochs drop out of displacements in the subsequent section too.

In [ ]:
df.null_count()

### What each field means

| Column | Type | Meaning |
|---|---|---|
| `timestamp` | datetime (UTC) | Epoch of the position estimate. The stream is 1 Hz, i.e. one row per second, per station. |
| `east` | float | East displacement from the station's reference position, in meters. |
| `north` | float | North displacement, in meters. |
| `up` | float | Vertical (up) displacement, in meters. |
| `sig_ee` | float | 1-sigma uncertainty on `east`, in meters. |
| `sig_nn` | float | 1-sigma uncertainty on `north`, in meters. |
| `sig_uu` | float | 1-sigma uncertainty on `up`, in meters. |
| `q_channel` | int | Integer-encoded quality and processing status information for each position estimate. |
| `ingest_latency` | duration | Time between the observation epoch and its ingest by the server. |
| `processing_delay` | duration | Time taken to process the epoch. |
| `geosncl` | str | Compound stream identifier: station.network.channel-location (e.g. `P146.PW.LY_.00`). |

## 5. Derive Displacement

The position components combine into a single displacement magnitude which tells us how far the station sits from its reference position at each epoch. Horizontal (2D) uses east and north; total (3D) adds the vertical:

$$d_{2D} = \sqrt{east^2 + north^2} \qquad d_{3D} = \sqrt{east^2 + north^2 + up^2}$$

In [ ]:
disp = df.with_columns(
    (pl.col("east").pow(2) + pl.col("north").pow(2)).sqrt().alias("disp_2d"),
    (pl.col("east").pow(2) + pl.col("north").pow(2) + pl.col("up").pow(2)).sqrt().alias("disp_3d"),
)

disp.select(["timestamp", "geosncl", "east", "north", "up", "disp_2d", "disp_3d"]).head()

Note that the square root propagates nulls in polar: if `east` or `north` is missing for an epoch, that epoch's `disp_2d` and `disp_3d` are null too. Drop them before computing statistics if needed using:

```python
disp_clean = disp.drop_nulls(subset=["disp_2d"])
```

> **Check:** For a quiet station, horizontal displacement (`disp_2d`) should stay small and roughly steady i.e. a few centimeters of PPP scatter around the reference. A sudden *step* in this value over time is what ground motion (e.g. an earthquake) would look like.

## 6. Visualize Position Streams & Displacement

### The position components over time

Reshape one station's `east` / `north` / `up` into long form and plot them together to see the three streams at once.

In [ ]:
one_long = df.filter(pl.col("geosncl").str.starts_with(STATIONS[0])).unpivot(
    ["east", "north", "up"],
    index="timestamp",
    variable_name="component",
    value_name="meters",
)

one_long.plot.line(x="timestamp", y="meters", color="component").properties(
    width=800, height=300, title=f"{STATIONS[0]}: position components over time"
)

### Displacement over time

Plot the 2D horizontal displacement for every station together. A quiet station traces a roughly flat, noisy band; a step would signal real motion.

In [ ]:
disp.plot.line(x="timestamp", y="disp_2d", color="geosncl").properties(
    width=800, height=300, title="2D horizontal displacement over time"
)

### A note on stream timeliness

The `ingest_latency` and `processing_delay` columns describe how *timely* the stream is, not how accurate the positions are. They are Polars `Duration` types and Altair can only plot numeric axes, so you must convert a Duration to a number (milliseconds) before plotting.

In [ ]:
# Altair plots only numeric axes, so convert the Duration column to milliseconds first.
df.with_columns(
    pl.col("ingest_latency").dt.total_milliseconds().alias("ingest_latency_ms")
).plot.line(x="timestamp", y="ingest_latency_ms", color="geosncl").properties(
    width=800, height=300, title="Ingest latency (ms)"
)

## 7. Exploration Exercises

Try modifying the parameters to explore how the results change.

1. **Different stations or window:** Change `STATIONS` and the `START`/`END` window in Configuration and re-run. Do all stations return a stream for your window?
2. **Show the uncertainty:** Plot `up` for one station with a shaded band of +/- `sig_uu` around it (hint: Altair's `mark_area` or `mark_errorband`). How wide is the vertical uncertainty compared to the signal?
3. **Noisiest station:** Compute the per-station standard deviation of `disp_2d` and identify which station is noisiest.

In [ ]:
# Exploration cell — use this space to experiment

## 8. Troubleshooting & Support

### Common Issues

| Error | Likely cause | Fix |
|---|---|---|
| Plotting error on a duration column | Altair cannot plot Duration types directly | Convert with `.dt.total_milliseconds()` before plotting |
| Null `disp_2d` / `disp_3d` values | `east` or `north` was null for that epoch | Use `drop_nulls(subset=["disp_2d"])` before computing statistics |

### Further Resources

* [EarthScope SDK Documentation](https://docs.earthscope.org/sdk)
* [GeoLab Documentation](https://docs.earthscope.org/geolab)
* [GeoLab Community Forum](https://earthscope.discourse.group/latest)